# Dim Customers

In [0]:
%sql
-- SUBQUERY

SELECT 
    *,
    row_number() OVER(order by customer_id) AS DimCustomerKey       -- Produces a unique number for all customers
FROM 
(
SELECT 
    DISTINCT(customer_id),   
    customer_email,
    customer_name,
    customer_name_upper
FROM datamodeling.silver.silver_table
)


customer_id,customer_email,customer_name,customer_name_upper,DimCustomerKey
1,alice@gmail.com,Alice Johnson,ALICE JOHNSON,1
2,bob@yahoo.com,Bob Smith,BOB SMITH,2
3,charlie@outlook.com,Charlie Brown,CHARLIE BROWN,3
4,david@hotmail.com,David Kim,DAVID KIM,4
5,eve@gmail.com,Eve Lee,EVE LEE,5


In [0]:
%sql
-- CTE (Common Table Expression)
CREATE OR REPLACE TABLE datamodeling.gold.Dim_Customers
AS
WITH remove_duplicate AS 
(
    SELECT DISTINCT
        customer_id,
        customer_email,
        customer_name,
        customer_name_upper
    FROM datamodeling.silver.silver_table
)
    SELECT 
        *,
        row_number() OVER(order by customer_id) AS DimCustomerKey
    FROM remove_duplicate


num_affected_rows,num_inserted_rows


# Dim Products

In [0]:
%sql
-- CTE (Common Table Expression)
CREATE OR REPLACE TABLE datamodeling.gold.Dim_Products
AS
WITH remove_duplicate AS 
(
    SELECT DISTINCT
        product_id,
        product_name,
        product_category
    FROM datamodeling.silver.silver_table
)
    SELECT 
        *,
        row_number() OVER(order by product_id) AS DimProductKey
    FROM remove_duplicate


num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT * 
FROM datamodeling.gold.Dim_Products

product_id,product_name,product_category,DimProductKey
501,iPhone 14,Electronics,1
502,AirPods Pro,Electronics,2
503,Nike Shoes,Footwear,3
504,Samsung Galaxy S23,Electronics,4
505,Apple Watch,Electronics,5


# Dim Payments

In [0]:
%sql
-- CTE
CREATE OR REPLACE TABLE datamodeling.gold.Dim_Payments
AS 
WITH remove_duplicate AS
(
    SELECT DISTINCT   
        payment_type
    FROM
        datamodeling.silver.silver_table
)
(
    SELECT 
        *,
        row_number() OVER (ORDER BY payment_type) AS DimPaymentKey
    FROM remove_duplicate
)


num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT *
FROM datamodeling.gold.Dim_Payments

payment_type,DimPaymentKey
Apple Pay,1
Credit Card,2
Google Pay,3
Paypal,4


# Dim Region

In [0]:
%sql
CREATE OR REPLACE TABLE datamodeling.gold.Dim_Region
AS
WITH remove_duplicate AS
(
    SELECT DISTINCT
        country
    FROM
        datamodeling.silver.silver_table
)
(
    SELECT 
        *,
        row_number() OVER (ORDER BY country) AS DimRegionKey
    FROM
        remove_duplicate
)

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT *
FROM datamodeling.gold.Dim_Region

country,DimRegion
Canada,1
USA,2


# Dim Sales (For Orders)

In [0]:
%sql
CREATE OR REPLACE TABLE datamodeling.gold.Dim_Sales 
AS
SELECT
    row_number() OVER (ORDER BY order_id) AS DimSaleKey,
    order_id,
    order_date,
    customer_id,
    customer_name,
    customer_email,
    product_id,
    product_name,
    product_category,
    payment_type,
    country,
    last_updated,
    customer_name_upper,
    process_timestamp
FROM 
    datamodeling.silver.silver_table

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT * FROM datamodeling.gold.Dim_Sales

DimSaleKey,order_id,order_date,customer_id,customer_name,customer_email,product_id,product_name,product_category,payment_type,country,last_updated,customer_name_upper,process_timestamp
1,1001,2024-07-01,1,Alice Johnson,alice@gmail.com,501,iPhone 14,Electronics,Credit Card,USA,2024-07-01,ALICE JOHNSON,2026-07-12
2,1002,2024-07-01,2,Bob Smith,bob@yahoo.com,502,AirPods Pro,Electronics,Paypal,USA,2024-07-01,BOB SMITH,2026-07-12
3,1003,2024-07-01,3,Charlie Brown,charlie@outlook.com,503,Nike Shoes,Footwear,Credit Card,Canada,2024-07-01,CHARLIE BROWN,2026-07-12
4,1004,2024-07-02,4,David Kim,david@hotmail.com,504,Samsung Galaxy S23,Electronics,Apple Pay,USA,2024-07-02,DAVID KIM,2026-07-12
5,1005,2024-07-02,5,Eve Lee,eve@gmail.com,505,Apple Watch,Electronics,Google Pay,USA,2024-07-02,EVE LEE,2026-07-12


# FACT TABLE (Measures and Foreign Keys to others Dimension Tables)

In [0]:
%sql
CREATE OR REPLACE TABLE datamodeling.gold.FactSales
AS
SELECT 
    S.DimSaleKey,
    C.DimCustomerKey,
    P.DimProductKey,
    R.DimRegionKey,
    PY.DimPaymentKey,
    
    -- numerical columns 
    F.quantity,
    F.unit_price,
    (F.quantity * F.unit_price) AS total_price
FROM
    datamodeling.silver.silver_table F           -- F for Fact table
LEFT JOIN  
    datamodeling.gold.Dim_Customers C
    ON F.customer_id = C.customer_id
LEFT JOIN
    datamodeling.gold.Dim_Products P
    ON F.product_id = P.product_id
LEFT JOIN
    datamodeling.gold.Dim_Region R
    ON F.country = R.country
LEFT JOIN
    datamodeling.gold.Dim_Payments PY
    ON F.payment_type = PY.payment_type
LEFT JOIN
    datamodeling.gold.Dim_Sales S
    ON F.order_id = S.order_id


num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT * 
FROM datamodeling.gold.FactSales

DimSaleKey,DimCustomerKey,DimProductKey,DimRegionKey,DimPaymentKey,quantity,unit_price,total_price
1,1,1,2,2,1,999.99,999.99
2,2,2,2,4,2,199.99,399.98
3,3,3,1,2,1,129.99,129.99
4,4,4,2,1,1,799.99,799.99
5,5,5,2,3,1,399.99,399.99
